# F2-M05: Análisis Univariante Categórico

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## Qué hace
Análisis univariante de todas las variables categóricas: clasificación por
cardinalidad (baja/media/alta), distribuciones con gráficos adaptados al
tipo (pie, barras, top-N), análisis geográfico (países, provincias,
poblaciones) y tabla resumen.

## Requisitos
- `df_alumno.parquet` en `data/02_processed/`
- Módulos: `src.config`, `src.utils`, `src.html`

## Genera
- `docs/html/fase2/m05_univariante_cat.html`
- `docs/html/fase2/graficos/m05_cardinalidad.html`
- `docs/html/fase2/graficos/m05_paises_*.html`
- `docs/html/fase2/graficos/m05_provincias_*.html`
- `docs/html/fase2/graficos/m05_poblaciones.html`
- `docs/html/fase2/graficos/m05_pie_*.html`
- `docs/html/fase2/graficos/m05_barras_*.html`
- `docs/html/fase2/graficos/m05_topn_*.html`

## Flujo
```
... → M04 Univariante Num → **M05 Univariante Cat** → M06 Evolución → ...
```

## Siguiente
`f2_m06_evolucion.ipynb`

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN DEL ENTORNO
# ============================================================================
# - Detecta entorno (Colab / local)
# - Localiza ROOT buscando src/ (robusto, sin hardcodes)
# - Importa módulos del proyecto y librerías de visualización
# - Crea directorios de salida
# ============================================================================

import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# --- Detectar entorno y localizar ROOT ---
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent
    _cwd = Path.cwd()
    ROOT = _cwd
    for _ in range(10):
        if (ROOT / 'src').is_dir():
            break
        ROOT = ROOT.parent
    else:
        raise FileNotFoundError(
            f"No se encontró carpeta 'src/' desde {_cwd}. "
            f"Verifica que el notebook está dentro de AU_UJI/"
        )

sys.path.insert(0, str(ROOT))

# --- Imports ---
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.config import RUTA_PROCESSED, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.html import (
    generar_kpis_html,
    generar_seccion_html,
    generar_html_navegacion_completa,
    guardar_html
)
from src.html.render import render_base_html

# --- Rutas de salida ---
RUTA_FASE2 = RUTA_HTML / 'fase2'
RUTA_GRAFICOS = RUTA_FASE2 / 'graficos'
crear_directorios([RUTA_FASE2, RUTA_GRAFICOS])

info_entorno()

✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATOS, DICCIONARIOS Y FUNCIONES AUXILIARES
# ============================================================================
# - Diccionarios: mapea códigos a etiquetas legibles
# - Funciones auxiliares: acortar texto, partir en 2 líneas,
#   conteo dual (matrículas + alumnos), formatear título.
# - Clasifica variables por cardinalidad.
# ============================================================================

print('=' * 60)
print('ANÁLISIS UNIVARIANTE CATEGÓRICO')
print('=' * 60)

df = pd.read_parquet(RUTA_PROCESSED / 'df_alumno.parquet')

# ============================================================================
# DICCIONARIOS PARA ETIQUETAS
# ============================================================================
FORMA_PAGO = {'T': 'Tarjeta', 'N': 'Recibo', 'D': 'Domiciliado'}
RAMA = {'TE': 'Tecnología', 'EX': 'Ciencias Experimentales', 'SO': 'Ciencias Sociales', 
        'HU': 'Humanidades', 'SA': 'Ciencias de la Salud'}

# Títulos especiales para variables
TITULOS_VARIABLES = {
    'nombre': 'Tipo de Matrícula',
    'nuevo': 'Nuevo Ingreso',
    'forma_de_pago': 'Forma de Pago',
    'egresado': 'Egresado',
    'sexo': 'Sexo',
    'rama': 'Rama de Conocimiento',
    'via_acceso': 'Vía de Acceso',
    'via_acceso_exp': 'Vía de Acceso (Expedientes)',
    'nombre_trabajo': 'Situación Laboral',
    'cupo': 'Cupo de Acceso',
    'titulacion': 'Titulación',
    'id_pais': 'País (código)',
    'pais_domicilio': 'País de Domicilio',
    'pais_nombre': 'País de Origen',
    'universidad_origen': 'Universidad de Origen',
    'orden_preferencia': 'Orden de Preferencia',
    'provincia': 'Provincia',
    'poblacion': 'Población',
    'nombre_beca': 'Tipo de Beca',
    'seguro': 'Seguro',
    'tiene_beca': 'Tiene Beca',
    'vive_fuera': 'Vive Fuera'
}

CONTINENTES = {
    'España': 'Europa', 'Rumanía': 'Europa', 'Italia': 'Europa', 'Francia': 'Europa',
    'Alemania': 'Europa', 'Portugal': 'Europa', 'Reino Unido': 'Europa', 'Polonia': 'Europa',
    'Marruecos': 'África', 'Argelia': 'África', 'Nigeria': 'África', 'Senegal': 'África',
    'Colombia': 'América', 'Ecuador': 'América', 'Argentina': 'América', 'México': 'América',
    'Perú': 'América', 'Venezuela': 'América', 'Chile': 'América', 'Brasil': 'América',
    'China': 'Asia', 'Pakistán': 'Asia', 'India': 'Asia', 'Rusia': 'Europa'
}

PROVINCIAS_CV = ['castelló', 'castellon', 'valencia', 'valèn', 'alicante', 'alacant']

DICCIONARIOS = {
    'forma_de_pago': FORMA_PAGO,
    'rama': RAMA
}

def aplicar_etiqueta(valor, col):
    if col in DICCIONARIOS:
        return DICCIONARIOS[col].get(str(valor), str(valor))
    return str(valor)

def get_titulo(col):
    return TITULOS_VARIABLES.get(col, col)

def acortar(texto, max_len=20):
    texto = str(texto)
    return texto[:max_len-2] + '..' if len(texto) > max_len else texto

def texto_dos_lineas(texto, max_len=18):
    texto = str(texto)
    if len(texto) <= max_len:
        return texto
    medio = len(texto) // 2
    espacio = texto.rfind(' ', 0, medio + 5)
    if espacio == -1:
        espacio = texto.find(' ', medio)
    if espacio != -1:
        return texto[:espacio] + '<br>' + texto[espacio+1:]
    return texto[:max_len] + '<br>' + texto[max_len:]

def contar_dual(df, col, id_col='per_id_ficticio'):
    mat = df[col].value_counts()
    alu = df.groupby(col)[id_col].nunique() if id_col in df.columns else mat
    return mat, alu


def abreviar_titulacion(nombre):
    """Abrevia nombres de titulación: Grado→G., Máster→M., etc."""
    t = str(nombre)
    t = t.replace('Grado en ', 'G. ')
    t = t.replace('Grau en ', 'G. ')
    t = t.replace('Máster Universitario en ', 'M. ')
    t = t.replace('Máster en ', 'M. ')
    t = t.replace('Master en ', 'M. ')
    t = t.replace('Doble Grado en ', 'DG. ')
    t = t.replace('Doble Grau en ', 'DG. ')
    t = t.replace('Ingeniería', 'Ing.')
    t = t.replace('Enginyeria', 'Ing.')
    t = t.replace('Administración', 'Adm.')
    t = t.replace('Comunicación', 'Com.')
    t = t.replace('Internacional', 'Int.')
    t = t.replace('Tecnología', 'Tecn.')
    t = t.replace('Informática', 'Inform.')
    t = t.replace(' y ', ' y ')
    t = t.replace(' de la ', ' ')
    t = t.replace(' de las ', ' ')
    t = t.replace(' de los ', ' ')
    t = t.replace(' del ', ' ')
    t = t.replace(' de ', ' ')
    # Máximo 35 chars
    if len(t) > 35:
        t = t[:33] + '..'
    return t

excluir = ['per_id_ficticio', 'exp_tit_id']
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
cat_cols = [c for c in cat_cols if c not in excluir]

print(f'📊 Variables categóricas: {len(cat_cols)}')

ANÁLISIS UNIVARIANTE CATEGÓRICO


📊 Variables categóricas: 17


In [3]:
# ============================================================================
# CELDA 3: CLASIFICAR POR CARDINALIDAD
# ============================================================================
# Clasifica cada variable categórica según su nº de valores únicos:
# - Baja (≤5): se representa con Pie
# - Media (6-15): se representa con Barras
# - Alta (>15): se representa con Top-N barras
# ============================================================================

clasificacion = []
for col in cat_cols:
    n = df[col].nunique()
    clasificacion.append({
        'Variable': col, 'Únicos': n,
        'Tipo': '🟢 Pie' if n <= 5 else '🔵 Barras' if n <= 15 else '🟠 Top-N'
    })

df_clasif = pd.DataFrame(clasificacion).sort_values('Únicos')

baja_card = [r['Variable'] for _, r in df_clasif.iterrows() if r['Únicos'] <= 5]
media_card = [r['Variable'] for _, r in df_clasif.iterrows() if 5 < r['Únicos'] <= 15]
alta_card = [r['Variable'] for _, r in df_clasif.iterrows() if r['Únicos'] > 15]

print(f'🟢 Baja (≤5): {len(baja_card)}')
print(f'🔵 Media (6-15): {len(media_card)}')
print(f'🟠 Alta (>15): {len(alta_card)}')

🟢 Baja (≤5): 6
🔵 Media (6-15): 3
🟠 Alta (>15): 8


In [4]:
# ============================================================================
# CELDA 4: GRÁFICO CARDINALIDAD (SIN CÍRCULOS)
# ============================================================================

print('\n' + '=' * 60)
print('GENERANDO GRÁFICOS')
print('=' * 60)

card_data = df_clasif.sort_values('Únicos', ascending=True)
colores_card = ['#38a169' if u <= 5 else '#3182ce' if u <= 15 else '#ed8936' for u in card_data['Únicos']]

fig_card = go.Figure(go.Bar(
    x=card_data['Únicos'], y=card_data['Variable'],
    orientation='h', marker_color=colores_card,
    text=card_data['Únicos'], textposition='outside'
))
fig_card.update_layout(
    title='📊 Cardinalidad de Variables Categóricas<br>'
          '<span style="font-size:11px; color:#718096;">'
          '🟢 Baja (≤5) | 🔵 Media (6-15) | 🟠 Alta (>15)</span>',
    xaxis_title='Valores Únicos',
    height=max(400, len(card_data)*28),
    margin=dict(t=80, b=50, l=180, r=80)
)
fig_card.write_html(RUTA_GRAFICOS / 'm05_cardinalidad.html', include_plotlyjs='cdn')
print('✅ Gráfico cardinalidad')


GENERANDO GRÁFICOS


✅ Gráfico cardinalidad


In [5]:
# ============================================================================
# CELDA 5: GRÁFICO PAÍSES - PIE (las barras irán en modal)
# ============================================================================

if 'pais_nombre' in df.columns:
    print('\n🌍 Países...')
    pais_counts = df['pais_nombre'].value_counts()
    df_pais = pd.DataFrame([{'pais': p, 'continente': CONTINENTES.get(p, 'Otros'), 'count': c} 
                           for p, c in pais_counts.items()])
    
    espana = df_pais[df_pais['pais'] == 'España']['count'].sum()
    otros = df_pais[df_pais['pais'] != 'España']
    
    # PIE principal
    fig_pais_pie = go.Figure(go.Pie(
        labels=['España', 'Otros países'],
        values=[espana, otros['count'].sum()],
        marker_colors=['#3182ce', '#ed8936'],
        textinfo='label+percent', hole=0.4,
        textposition='outside',
        rotation=90,
        hovertemplate='%{label}<br>%{value:,} (%{percent})<extra></extra>'
    ))
    fig_pais_pie.update_layout(
        title='🌍 País de Origen<br>'
              '<span style="font-size:11px; color:#718096;">'
              'Click para ver Top 15 países extranjeros</span>',
        height=420, margin=dict(t=80, b=40, l=40, r=40),
        showlegend=False
    )
    fig_pais_pie.write_html(RUTA_GRAFICOS / 'm05_paises_pie.html', include_plotlyjs='cdn')
    
    # BARRAS para modal
    top_paises = otros.nlargest(15, 'count').sort_values('count', ascending=True)
    colores_cont = {'Europa': '#3182ce', 'América': '#38a169', 'África': '#ed8936', 'Asia': '#e53e3e', 'Otros': '#718096'}
    colores_barras = [colores_cont.get(c, '#718096') for c in top_paises['continente']]
    
    fig_pais_bar = go.Figure(go.Bar(
        x=top_paises['count'], y=top_paises['pais'],
        orientation='h', marker_color=colores_barras,
        text=[f'{v:,}' for v in top_paises['count']], textposition='outside'
    ))
    fig_pais_bar.update_layout(
        title='🌍 Top 15 Países (sin España)',
        xaxis_title='Matrículas',
        height=450, margin=dict(t=60, b=50, l=120, r=80)
    )
    fig_pais_bar.write_html(RUTA_GRAFICOS / 'm05_paises_barras.html', include_plotlyjs='cdn')
    print('✅ Gráficos países (pie + barras para modal)')


🌍 Países...
✅ Gráficos países (pie + barras para modal)


In [6]:
# ============================================================================
# CELDA 6: GRÁFICO PROVINCIAS - PIE (las barras irán en modal)
# ============================================================================

if 'provincia' in df.columns:
    print('\n📍 Provincias...')
    prov_counts = df['provincia'].value_counts()
    
    cv_total, otras_total = 0, 0
    cv_detalle = {}
    
    for p, c in prov_counts.items():
        es_cv = any(cv in str(p).lower() for cv in PROVINCIAS_CV)
        if es_cv:
            cv_total += c
            cv_detalle[p] = c
        else:
            otras_total += c
    
    # PIE principal
    fig_prov_pie = go.Figure(go.Pie(
        labels=['Comunitat Valenciana', 'Otras'],
        values=[cv_total, otras_total],
        marker_colors=['#3182ce', '#ed8936'],
        textinfo='label+percent', hole=0.4,
        textposition='outside',
        rotation=90,
        hovertemplate='%{label}<br>%{value:,} (%{percent})<extra></extra>'
    ))
    fig_prov_pie.update_layout(
        title='📍 Provincia<br>'
              '<span style="font-size:11px; color:#718096;">'
              'Click para ver detalle Comunitat Valenciana</span>',
        height=420, margin=dict(t=80, b=40, l=40, r=40),
        showlegend=False
    )
    fig_prov_pie.write_html(RUTA_GRAFICOS / 'm05_provincias_pie.html', include_plotlyjs='cdn')
    
    # BARRAS para modal
    cv_df = pd.Series(cv_detalle).sort_values(ascending=True)
    fig_prov_bar = go.Figure(go.Bar(
        x=cv_df.values, y=cv_df.index,
        orientation='h', marker_color='#3182ce',
        text=[f'{v:,} ({v/cv_total*100:.1f}%)' for v in cv_df.values], textposition='outside'
    ))
    fig_prov_bar.update_layout(
        title='📍 Detalle Comunitat Valenciana',
        xaxis_title='Matrículas',
        height=350, margin=dict(t=60, b=50, l=120, r=100)
    )
    fig_prov_bar.write_html(RUTA_GRAFICOS / 'm05_provincias_barras.html', include_plotlyjs='cdn')    
    # Barras de provincias FUERA de CV (para segundo modal)
    otras_prov = {}
    for p, c in prov_counts.items():
        es_cv = any(cv in str(p).lower() for cv in PROVINCIAS_CV)
        if not es_cv:
            otras_prov[p] = c
    
    if otras_prov:
        otras_df = pd.Series(otras_prov).sort_values(ascending=True).tail(15)
        fig_otras = go.Figure(go.Bar(
            x=otras_df.values, y=otras_df.index,
            orientation='h', marker_color='#ed8936',
            text=[f'{v:,} ({v/otras_total*100:.1f}%)' for v in otras_df.values],
            textposition='outside'
        ))
        fig_otras.update_layout(
            title='📍 Top 15 Provincias (fuera de CV)',
            xaxis_title='Matrículas',
            height=450, margin=dict(t=60, b=50, l=160, r=100)
        )
        fig_otras.update_xaxes(range=[0, otras_df.max() * 1.3])
        fig_otras.write_html(RUTA_GRAFICOS / 'm05_provincias_otras.html', include_plotlyjs='cdn')
        print('✅ Gráfico provincias otras (fuera CV)')

    print('✅ Gráficos provincias (pie + barras para modal)')


📍 Provincias...


✅ Gráfico provincias otras (fuera CV)
✅ Gráficos provincias (pie + barras para modal)


In [7]:
# ============================================================================
# CELDA 7: TREEMAP POBLACIONES (TONOS POR %)
# ============================================================================

if 'poblacion' in df.columns:
    print('\n🏘️ Treemap Poblaciones...')
    pobl_counts = df['poblacion'].value_counts().head(30)
    total_pobl = pobl_counts.sum()
    
    def detectar_prov(pobl):
        p = str(pobl).lower()
        if 'castelló' in p or 'castello' in p or 'vila-real' in p or 'borriana' in p or 'onda' in p or 'benicàssim' in p:
            return 'Castellón'
        elif 'valèn' in p or 'valen' in p or 'sagunt' in p:
            return 'Valencia'
        elif 'alicant' in p or 'alacant' in p:
            return 'Alicante'
        return 'Otras'
    
    df_pobl = pd.DataFrame([{
        'poblacion': acortar(p, 18), 
        'provincia': detectar_prov(p), 
        'count': c,
        'pct': c / total_pobl * 100  # Para tonos
    } for p, c in pobl_counts.items()])
    
    # Treemap con tonos por porcentaje
    fig_pobl = px.treemap(
        df_pobl, 
        path=['provincia', 'poblacion'], 
        values='count',
        color='pct',  # Tonos por porcentaje
        color_continuous_scale='Blues',
        title='🏘️ Top 30 Poblaciones (tonos por %)'
    )
    fig_pobl.update_layout(height=500, margin=dict(t=60, b=40, l=40, r=40))
    fig_pobl.update_traces(
        textinfo='label+percent entry',
        hovertemplate='<b>%{label}</b><br>Matrículas: %{value:,}<br>%{percentEntry:.1%}<extra></extra>'
    )
    fig_pobl.write_html(RUTA_GRAFICOS / 'm05_poblaciones.html', include_plotlyjs='cdn')
    print('✅ Treemap poblaciones con tonos')


🏘️ Treemap Poblaciones...

✅ Treemap poblaciones con tonos


In [8]:
# ============================================================================
# CELDA 8: GRÁFICOS BAJA CARDINALIDAD (PIE) - CON DICCIONARIOS
# ============================================================================

print('\n📊 Gráficos baja cardinalidad...')
graficos_pie = []

for col in baja_card:
    mat, _ = contar_dual(df, col)
    # Mapear S/N a Sí/No para variables binarias
    SN_MAP = {'S': 'Sí', 'N': 'No'}
    labels = []
    for x in mat.index:
        val = aplicar_etiqueta(x, col)
        val = SN_MAP.get(str(val), val)
        labels.append(val)
    
    fig = go.Figure(go.Pie(
        labels=labels,
        values=mat.values, hole=0.4,
        textinfo='label+percent', textposition='outside'
    ))
    titulo = get_titulo(col)
    fig.update_layout(title=f'📊 {titulo}', height=380,
                      margin=dict(t=60, b=40, l=40, r=40), showlegend=False)
    nombre = f'm05_pie_{col}.html'
    fig.write_html(RUTA_GRAFICOS / nombre, include_plotlyjs='cdn')
    graficos_pie.append(nombre)

print(f'✅ {len(graficos_pie)} gráficos pie')


📊 Gráficos baja cardinalidad...


✅ 6 gráficos pie


In [9]:
# ============================================================================
# CELDA 9: GRÁFICOS MEDIA CARDINALIDAD (BARRAS)
# ============================================================================

print('\n📊 Gráficos media cardinalidad...')
graficos_barras = []
graficos_barras_solos = []

for col in media_card:
    mat, _ = contar_dual(df, col)
    mat = mat.sort_values(ascending=True)
    
    if col in ['via_acceso', 'nombre_trabajo']:
        labels = [acortar(aplicar_etiqueta(x, col), 30) for x in mat.index]
        bar_height = max(480, len(mat)*45)
        margen_izq = 220
    else:
        labels = [acortar(aplicar_etiqueta(x, col), 20) for x in mat.index]
        bar_height = max(350, len(mat)*30)
        margen_izq = 180
    
    pct = (mat / mat.sum() * 100).values
    
    fig = go.Figure(go.Bar(
        x=mat.values, y=labels, orientation='h',
        marker_color='#3182ce',
        text=[f'{v:,} ({p:.1f}%)' for v, p in zip(mat.values, pct)],
        textposition='outside',
        textfont=dict(size=10)
    ))
    
    titulo = get_titulo(col)
    fig.update_layout(
        title=f'📊 {titulo}',
        xaxis_title='Frecuencia',
        height=bar_height,
        margin=dict(t=60, b=50, l=margen_izq, r=100),
        bargap=0.15 if col in ['via_acceso', 'nombre_trabajo'] else 0.2
    )
    fig.update_xaxes(range=[0, mat.max() * 1.65])
    
    nombre = f'm05_barras_{col}.html'
    fig.write_html(RUTA_GRAFICOS / nombre, include_plotlyjs='cdn')
    
    if 'titulacion' in col.lower():
        graficos_barras_solos.append(nombre)
    else:
        graficos_barras.append(nombre)

print(f'✅ {len(graficos_barras)} en pares + {len(graficos_barras_solos)} solos')


📊 Gráficos media cardinalidad...
✅ 3 en pares + 0 solos


In [10]:
# ============================================================================
# CELDA 10: GRÁFICOS ALTA CARDINALIDAD (TOP-15 o TODAS)
# ============================================================================
# - Titulación: TODAS las barras dobles (matrículas + alumnos únicos)
#   con nombres abreviados (G.=Grado, M.=Máster, Ing.=Ingeniería)
# - Resto: Top 15 barras simples con títulos legibles
# ============================================================================

print('\n📊 Gráficos alta cardinalidad...')
graficos_topn = []
graficos_topn_solos = []
especiales = ['pais_nombre', 'provincia', 'poblacion', 'pais_domicilio', 'id_pais']

for col in alta_card:
    if col in especiales:
        continue
    
    titulo = get_titulo(col)
    mat, alu = contar_dual(df, col)
    
    if 'titulacion' in col.lower():
        # --- TITULACIÓN: TODAS + barras dobles ---
        mat_sorted = mat.sort_values(ascending=True)
        alu_sorted = alu.reindex(mat_sorted.index).fillna(0).astype(int)
        labels = [abreviar_titulacion(x) for x in mat_sorted.index]
        n_barras = len(mat_sorted)
        
        fig = go.Figure()
        fig.add_trace(go.Bar(
            x=mat_sorted.values, y=labels, orientation='h',
            marker_color='#E040FB', opacity=0.6,
            name='Matrículas',
            text=[f'{v:,}' for v in mat_sorted.values],
            textposition='outside',
            hovertemplate='%{y}<br>Matrículas: %{x:,}<extra></extra>'
        ))
        fig.add_trace(go.Bar(
            x=alu_sorted.values, y=labels, orientation='h',
            marker_color='#3182ce', opacity=0.8,
            name='Alumnos únicos',
            text=[f'{v:,}' for v in alu_sorted.values],
            textposition='outside',
            hovertemplate='%{y}<br>Alumnos únicos: %{x:,}<extra></extra>'
        ))
        fig.update_layout(
            title='📊 Titulación (todas)<br>'
                  '<span style="font-size:11px; color:#718096;">'
                  '🟠 Matrículas totales | 🔵 Alumnos únicos</span>',
            xaxis_title='Frecuencia',
            barmode='group',
            height=max(700, n_barras * 32),
            margin=dict(t=80, b=50, l=280, r=100),
            legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='center', x=0.5)
        )
        fig.update_xaxes(range=[0, mat_sorted.max() * 1.25])
    else:
        # --- RESTO: Top 15 barras simples ---
        top_data = mat.head(15).sort_values(ascending=True)
        labels = [acortar(aplicar_etiqueta(x, col), 22) for x in top_data.index]
        pct = (top_data / mat.sum() * 100).values
        n_barras = len(top_data)
        
        fig = go.Figure(go.Bar(
            x=top_data.values, y=labels, orientation='h',
            marker_color='#ed8936',
            text=[f'{v:,} ({p:.1f}%)' for v, p in zip(top_data.values, pct)],
            textposition='outside',
            textfont=dict(size=10)
        ))
        fig.update_layout(
            title=f'🏆 Top 15: {titulo}',
            xaxis_title='Frecuencia',
            height=max(450, n_barras * 30),
            margin=dict(t=60, b=50, l=200, r=100)
        )
        fig.update_xaxes(range=[0, top_data.max() * 1.65])
    
    nombre = f'm05_topn_{col}.html'
    fig.write_html(RUTA_GRAFICOS / nombre, include_plotlyjs='cdn')
    
    if 'titulacion' in col.lower():
        graficos_topn_solos.append(nombre)
    else:
        graficos_topn.append(nombre)

print(f'✅ {len(graficos_topn)} en pares + {len(graficos_topn_solos)} solos')



📊 Gráficos alta cardinalidad...


✅ 2 en pares + 1 solos


In [11]:
# ============================================================================
# CELDA 11: TABLA RESUMEN DE VARIABLES
# ============================================================================

print('\n📋 Preparando tabla resumen...')

resumen = []
for col in cat_cols:
    mat = df[col].value_counts()
    top1 = mat.index[0] if len(mat) > 0 else ''
    top1_pct = mat.iloc[0] / len(df) * 100 if len(mat) > 0 else 0
    moda_label = aplicar_etiqueta(top1, col)
    
    resumen.append({
        'Variable': col,
        'Únicos': df[col].nunique(),
        'Nulos': df[col].isnull().sum(),
        '%Nulos': df[col].isnull().mean() * 100,
        'Moda': acortar(moda_label, 20),
        '%Moda': top1_pct
    })

df_resumen = pd.DataFrame(resumen).sort_values('Únicos')
print(f'✅ Tabla resumen: {len(df_resumen)} variables')


📋 Preparando tabla resumen...


✅ Tabla resumen: 17 variables


In [12]:
# ============================================================================
# CELDA 12: GENERAR HTML CON MODALES POPUP
# ============================================================================

print('\n' + '=' * 60)
print('GENERANDO HTML')
print('=' * 60)

nav_fases_html, nav_modulos_html = generar_html_navegacion_completa(
    fase_activa="fase2", modulo_activo="m05"
)

KPIS = [
    {'valor': str(len(cat_cols)), 'titulo': 'Variables Categóricas'},
    {'valor': str(len(baja_card)), 'titulo': '🟢 Baja (≤5)'},
    {'valor': str(len(media_card)), 'titulo': '🔵 Media (6-15)'},
    {'valor': str(len(alta_card)), 'titulo': '🟠 Alta (>15)'},
]
kpis_html = generar_kpis_html(KPIS)

# CSS para modales
modal_css = '''
<style>
.modal-overlay {
    display: none;
    position: fixed;
    top: 0; left: 0;
    width: 100%; height: 100%;
    background: rgba(0,0,0,0.7);
    z-index: 1000;
    justify-content: center;
    align-items: center;
}
.modal-content {
    background: white;
    border-radius: 10px;
    width: 90%;
    max-width: 800px;
    max-height: 90%;
    overflow: auto;
}
.modal-header {
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding: 15px 20px;
    border-bottom: 1px solid #e2e8f0;
}
.modal-close {
    background: none;
    border: none;
    font-size: 1.5em;
    cursor: pointer;
    color: #718096;
}
.modal-close:hover { color: #e53e3e; }
.clickable-chart {
    cursor: pointer;
    transition: transform 0.2s;
}
.clickable-chart:hover {
    transform: scale(1.02);
    box-shadow: 0 4px 12px rgba(0,0,0,0.15);
}
</style>
'''

def fila_2(g1, g2=None, h=420):
    if g2:
        return f'''<div style="display:grid; grid-template-columns:1fr 1fr; gap:20px; margin-bottom:20px;">
            <iframe src="graficos/{g1}" width="100%" height="{h}" frameborder="0"></iframe>
            <iframe src="graficos/{g2}" width="100%" height="{h}" frameborder="0"></iframe>
        </div>'''
    return f'<iframe src="graficos/{g1}" width="100%" height="{h}" frameborder="0" style="margin-bottom:20px;"></iframe>'

# Sección cardinalidad
sec_card = generar_seccion_html(
    titulo="Cardinalidad de Variables",
    contenido=f'<iframe src="graficos/m05_cardinalidad.html" width="100%" height="{max(400, len(cat_cols)*28)}" frameborder="0"></iframe>',
    icono="📊"
)

# Sección geográfica con MODALES
cont_geo = '''
<div style="display:grid; grid-template-columns:1fr 1fr; gap:20px; margin-bottom:20px;">
    <div class="clickable-chart" onclick="document.getElementById('modal-paises').style.display='flex'">
        <iframe src="graficos/m05_paises_pie.html" width="100%" height="420" frameborder="0" style="pointer-events:none;"></iframe>
    </div>
    <div class="clickable-chart" onclick="document.getElementById('modal-provincias').style.display='flex'">
        <iframe src="graficos/m05_provincias_pie.html" width="100%" height="420" frameborder="0" style="pointer-events:none;"></iframe>
    </div>
</div>

<!-- Modal Países -->
<div id="modal-paises" class="modal-overlay" onclick="if(event.target==this)this.style.display='none'">
    <div class="modal-content">
        <div class="modal-header">
            <h3 style="margin:0;">🌍 Top 15 Países (sin España)</h3>
            <button class="modal-close" onclick="document.getElementById('modal-paises').style.display='none'">×</button>
        </div>
        <iframe src="graficos/m05_paises_barras.html" width="100%" height="480" frameborder="0"></iframe>
    </div>
</div>

<!-- Modal Provincias -->
<div id="modal-provincias" class="modal-overlay" onclick="if(event.target==this)this.style.display='none'">
    <div class="modal-content" style="max-width:1200px;">
        <div class="modal-header">
            <h3 style="margin:0;">📍 Provincias: Comunitat Valenciana + Resto</h3>
            <button class="modal-close" onclick="document.getElementById('modal-provincias').style.display='none'">×</button>
        </div>
        <div style="display:grid; grid-template-columns:1fr 1fr; gap:10px; padding:10px;">
            <iframe src="graficos/m05_provincias_barras.html" width="100%" height="400" frameborder="0"></iframe>
            <iframe src="graficos/m05_provincias_otras.html" width="100%" height="400" frameborder="0"></iframe>
        </div>
    </div>
</div>
'''

# Poblaciones (sin modal)
if 'poblacion' in df.columns:
    cont_geo += fila_2('m05_poblaciones.html', h=520)

sec_geo = generar_seccion_html(titulo="Variables Geográficas", contenido=cont_geo, icono="🌍")

# Sección baja cardinalidad (Pie)
cont_pie = ''.join([fila_2(graficos_pie[i], graficos_pie[i+1] if i+1 < len(graficos_pie) else None, 400) 
                    for i in range(0, len(graficos_pie), 2)])
sec_pie = generar_seccion_html(titulo="Baja Cardinalidad (≤5)", contenido=cont_pie, icono="🟢") if graficos_pie else ''

# Sección media cardinalidad (Barras)
cont_bar = ''.join([fila_2(graficos_barras[i], graficos_barras[i+1] if i+1 < len(graficos_barras) else None, 500) 
                    for i in range(0, len(graficos_barras), 2)])
for g in graficos_barras_solos:
    cont_bar += fila_2(g, h=520)
sec_bar = generar_seccion_html(titulo="Media Cardinalidad (6-15)", contenido=cont_bar, icono="🔵") if (graficos_barras or graficos_barras_solos) else ''

# Sección alta cardinalidad (Top-N)
cont_topn = ''.join([fila_2(graficos_topn[i], graficos_topn[i+1] if i+1 < len(graficos_topn) else None, 470) 
                     for i in range(0, len(graficos_topn), 2)])
for g in graficos_topn_solos:
    cont_topn += fila_2(g, h=520)
sec_topn = generar_seccion_html(titulo="Alta Cardinalidad (>15) - Top 15", contenido=cont_topn, icono="🟠") if (graficos_topn or graficos_topn_solos) else ''

# Tabla resumen
tabla_html = df_resumen.to_html(classes='tabla-estadisticas', index=False, float_format=lambda x: f'{x:.1f}')
sec_tabla = generar_seccion_html(
    titulo="Resumen de Variables",
    contenido=f'<div style="overflow-x:auto; max-height:400px; overflow-y:auto;">{tabla_html}</div>',
    icono="📋"
)

contenido_html = modal_css + kpis_html + sec_card + sec_geo + sec_pie + sec_bar + sec_topn + sec_tabla

html_completo = render_base_html(
    titulo="M05: Univariante Cat",
    icono="📊",
    subtitulo="Fase 2: EDA Datos Originales | TFM Abandono Universitario",
    nav_fases=nav_fases_html,
    nav_modulos=nav_modulos_html,
    contenido=contenido_html,
    notebook_nombre='f2_m05_univariante_cat.ipynb',
    notebook_carpeta='fase2_eda'
)

ruta_html = RUTA_FASE2 / "m05_univariante_cat.html"
guardar_html(html_completo, ruta_html)
print(f"\n✅ HTML: {ruta_html}")


GENERANDO HTML
✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase2\m05_univariante_cat.html

✅ HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase2\m05_univariante_cat.html


In [13]:
# ============================================================================
# CELDA TEMPORAL: EXPORTAR MUESTRA A EXCEL (BORRAR DESPUÉS)
# ============================================================================

# Columnas de interés
cols_ver = ['per_id_ficticio', 'pais_nombre', 'pais_domicilio', 'id_pais', 
            'provincia', 'poblacion', 'titulacion', 'rama', 'sexo',
            'curso_aca', 'nota_acceso', 'egresado', 'nuevo']
cols_disponibles = [c for c in cols_ver if c in df.columns]

# Exportar 500 filas aleatorias a Excel
muestra = df[cols_disponibles].sample(500, random_state=42)
ruta_excel = RUTA_FASE2 / 'muestra_datos_m05.xlsx'
muestra.to_excel(ruta_excel, index=True)
print(f'✅ Excel exportado: {ruta_excel}')
print(f'   {len(muestra)} filas × {len(cols_disponibles)} columnas')

# Resumen rápido
print(f'\n--- Resumen de las variables geográficas ---')
for col in ['pais_nombre', 'pais_domicilio', 'id_pais']:
    if col in df.columns:
        n_u = df[col].nunique()
        top1 = df[col].value_counts().index[0]
        top1_pct = df[col].value_counts().iloc[0] / len(df) * 100
        print(f'  {col}: {n_u} únicos | Top: {top1} ({top1_pct:.1f}%)')

print(f'\n⚠️ pais_domicilio: {df["pais_domicilio"].value_counts().head(1).values[0] / len(df) * 100:.1f}% es España → NO ÚTIL')
print(f'⚠️ id_pais: redundante con pais_nombre (misma info, diferente formato)')
print(f'\n💡 Recomendación: excluir pais_domicilio e id_pais de los gráficos')


✅ Excel exportado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase2\muestra_datos_m05.xlsx
   500 filas × 13 columnas

--- Resumen de las variables geográficas ---
  pais_nombre: 77 únicos | Top: España (93.9%)
  pais_domicilio: 22 únicos | Top: España (99.7%)
  id_pais: 77 únicos | Top: E (93.9%)

⚠️ pais_domicilio: 99.7% es España → NO ÚTIL
⚠️ id_pais: redundante con pais_nombre (misma info, diferente formato)

💡 Recomendación: excluir pais_domicilio e id_pais de los gráficos


In [14]:
print('\n' + '=' * 60)
print('✅ F2-M05 COMPLETADO')
print('=' * 60)
print(f'📊 Pies: {len(graficos_pie)}')
print(f'📊 Barras: {len(graficos_barras)} pares + {len(graficos_barras_solos)} solos')
print(f'📊 Top-N: {len(graficos_topn)} pares + {len(graficos_topn_solos)} solos')
print(f'📋 Tabla resumen: {len(df_resumen)} variables')
print('🖱️ Click en pies de países/provincias → modal con barras')
print('📌 Siguiente: f2_m06_evolucion.ipynb')


✅ F2-M05 COMPLETADO
📊 Pies: 6
📊 Barras: 3 pares + 0 solos
📊 Top-N: 2 pares + 1 solos
📋 Tabla resumen: 17 variables
🖱️ Click en pies de países/provincias → modal con barras
📌 Siguiente: f2_m06_evolucion.ipynb
